# Session 8: It should know your policies

**Week 4 · Session 2 · Project version v5**

v4 is guarded: it keeps customers inside their own orders and every refund waits for a manager. But
it answers policy questions **from the model's memory**, and memory is not policy. Tonight we build
the thing it will look things up in.

<img src="figures/session-map.png" width="900"
     alt="The whole of session 8 on one page. Across the top, v4 answers policy from memory and v5
     has a knowledge base it can search. Below, two zones. Zone A, ingestion: markdown and PDF
     become one Document shape, chunk size is an experiment, and every chunk carries source,
     section and page. Zone B, embeddings and search, measured: meaning becomes coordinates, one
     model for the index and the query, a collection on disk with stable ids, and hit-rate and MRR
     before any answer. A bottom band says score retrieval first, because generation cannot fix
     what retrieval did not find.">

There are two main topics:

| | Topic | Question it answers |
|---|---|---|
| **A** | **Ingestion** | What goes into the knowledge base, and in what shape? |
| **B** | **Embeddings and search, measured** | How does it find things, and how do we *know* it found the right ones? |

By the end you will have **v5**: a knowledge base on disk, built by `kb/ingest.py`, with a
retrieval score. The session **stops before any answer is generated**, on purpose. Session 9 hires
the librarian, and a librarian can only be judged against a library you have already inspected.

## 1. Setup

Three dependencies are new to this session, and `uv sync` installed them already:

| Package | Job |
|---|---|
| `langchain-text-splitters` | chunking |
| `pypdf` | reading PDFs |
| `qdrant-client[fastembed]` | the vector database, plus small CPU models for keyword vectors and a keyless fallback |

The v5 project adds a `kb/` package. The notebook imports its real functions rather than copies:

| Module | What it does |
|---|---|
| `kb/store.py` | opens the knowledge base: the embedding model, the Qdrant client, the collection |
| `kb/ingest.py` | builds it: load → chunk → embed → write |
| `kb/retrieval_eval.py` | scores it against `data/kb_questions.yml` |

**Two providers tonight.** The chat model is still session 6's `openrouter:openai/gpt-oss-20b`,
used only in section 2. Embeddings come from **Gemini's `gemini-embedding-2`**, on the free tier,
with the `GEMINI_API_KEY` you set up in session 1. OpenRouter is a router for *chat* models; the
embedding model is a separate choice, made in `config.py`. No Gemini key? Section 9 shows the free
CPU fallback.

In [35]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
from langchain_core._api import LangChainBetaWarning
from langchain_core.documents import Document
from langgraph.checkpoint.memory import InMemorySaver

warnings.filterwarnings("ignore", category=LangChainBetaWarning)
warnings.filterwarnings("ignore", message="IProgress not found")    # fastembed's progress bar

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))        # the v5 snapshot beside this notebook
from config import Settings, settings             # noqa: E402
from kb import ingest, store                      # noqa: E402
from kb import retrieval_eval as scorer           # noqa: E402

POLICIES = settings.POLICIES_DIR
print(f"{settings.APP_NAME} v{settings.VERSION}")
print(f"chat model:      {settings.model_id}")
print(f"embedding space: {settings.EMBED_MODEL} · {settings.EMBED_DIM}")
print(f"collection:      {settings.collection}")

ShopWise Support Assistant v5.0.0
chat model:      openrouter:openai/gpt-oss-20b
embedding space: gemini-embedding-2 · 768
collection:      policies__gemini-embedding-2-768


`show` prints a function straight from a project file, so you read the code that actually runs.

In [36]:
def show(name, file="kb/ingest.py"):
    """Print one top-level definition from a project file: the real code, not a copy."""
    lines = (HERE / "project" / file).read_text().splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith((f"def {name}(", f"class {name}")))
    end = start + 1
    while end < len(lines) and (not lines[end] or lines[end][0] in " )]"):
        end += 1
    print("\n".join(lines[start:end]).rstrip())

## 2. The failure: policy from memory

Session 7 ended on Priya's question. Here it is, asked twice to the unchanged v4 agent, logged in as
Priya. ORD-5004 is her TrailPack backpack, delivered on 7 May.

In [37]:
from agent import answer, build_agent       # noqa: E402
import middleware as mw                     # noqa: E402

v4 = build_agent(InMemorySaver())
PRIYA = mw.customer_for("priya@example.com")
QUESTION = "Can I still return the backpack from order ORD-5004?"

for run in (1, 2):
    reply = answer(v4, QUESTION, f"T-{run}", PRIYA)["messages"][-1].text
    print(f"run {run}: {reply}\n")

run 1: I’m sorry, but the order was delivered on 2026‑05‑07, which is well beyond our standard return window. We can’t accept a return for that backpack. If you have any other questions or need assistance with a different issue, let me know.

run 2: I’m checking the return window for that order. Could you let me know the return policy period we offer for backpacks? That will help me confirm if it’s still eligible.



Now the document the model has never seen:

In [38]:
policy = (POLICIES / "returns-policy.md").read_text()
print(policy[policy.index("## 1."):policy.index("1.2")])

## 1. The return window

1.1 You may return most items within **30 days of the delivery date** shown in your order history.
The window is counted from delivery, not from the order date.




Read the two runs against that. The agent can look up the **delivery date**: that is a tool, and
session 5 gave it one. It cannot look up the **window**. So whatever it says about 30 days, 60 days or
"I don't have our return policy", it is either a guess or a polite non-answer, and you can't tell
which from the reply. Run it again and the answer may change.

A smarter model fixes neither. Giving it the document does. *We don't need a smarter model. We need a
librarian, and first, a library.*

## 3. What retrieval is

Two constraints make retrieval necessary:

| Constraint | Consequence |
|---|---|
| The context window is finite | you cannot paste every document into every prompt |
| The model's knowledge is frozen, and was never yours | it has not read ShopWise's policies, and never will by itself |

So we find the few passages that matter **at question time**, and hand only those to the model. That
is retrieval-augmented generation (RAG), and it is a pipeline:

<img src="figures/pipeline.png" width="900"
     alt="The retrieval pipeline in seven stages. The left half, built today in session 8, is load,
     chunk, embed and store, ending in the knowledge base. The right half, built in session 9, is
     query, retrieve and generate. Below, three ways to put the pipeline to work: 2-step, which
     always retrieves, and agentic, where the agent decides when to search, both in session 9; and
     loops that grade and retry, in session 10.">

Tonight is the left half: **load → chunk → embed → store**, then a check that searching it works.
Session 9 is the right half.

## 4. Prompt, RAG or fine-tune?

Before building, the question every junior engineer gets in their first month, from a manager or in
an interview: *"why not just fine-tune the model on our policies?"*

<img src="figures/prompt-rag-finetune.png" width="900"
     alt="A decision plate: prompt, RAG or fine-tune. Prompt when the knowledge fits in the prompt
     and changes often; cheapest, no training. RAG when the facts are yours, large and changing; add
     a document and it is known in seconds. Fine-tune when you need a consistent style, format or
     behaviour; it costs hundreds of labelled examples and a training run. A warning panel: a
     fine-tuned model still cannot tell you today's refund policy. The takeaway: fine-tuning teaches
     a model how to behave; retrieval teaches it what is true.">

| Choose | When | Cost of a change |
|---|---|---|
| **Prompt** | the knowledge fits in the prompt and changes often | edit text, live instantly |
| **RAG** | the facts are yours, large, and changing | add or edit a document, live in seconds |
| **Fine-tune** | you need a consistent **style, format or behaviour**, and have hundreds of labelled examples | a new dataset and a training run |

**Fine-tuning teaches a model how to behave; retrieval teaches it what is true.** A fine-tuned
model still can't tell you *today's* refund policy, and retraining on every policy edit is absurd.
ShopWise's policies change, so: RAG.

The honest caveat, so nobody leaves thinking fine-tuning is useless: it wins for a rigid output
format at very high volume, where the tokens saved by dropping a long prompt pay for the training
run. And you have already used the middle ground: session 3's few-shot examples. **Examples in the
prompt are fine-tuning's cheap cousin.**

---
# Part A: Ingestion

## 5. Loaders: one `Document` shape for every source

Whatever the file, a loader produces `Document`s: `page_content` (the text) plus `metadata` (where it
came from). Everything downstream (splitters, embeddings, vector stores) only ever sees that shape.

The same returns policy exists twice in `data/policies/`, as markdown and as PDF. Loaded both ways:

In [39]:
md_docs = ingest.load_markdown(POLICIES / "returns-policy.md")
pdf_docs = ingest.load_pdf(POLICIES / "returns-policy.pdf")

print(f"markdown -> {len(md_docs)} Documents, one per section:")
for d in md_docs:
    print("   ", d.metadata["section"])
print(f"\nPDF -> {len(pdf_docs)} Document, one per page:\n   ", pdf_docs[0].metadata)

markdown -> 5 Documents, one per section:
    preamble
    1. The return window
    2. What cannot be returned
    3. How refunds are paid
    4. Starting a return

PDF -> 1 Document, one per page:
    {'source': 'returns-policy', 'section': 'p.1', 'page': 1, 'format': 'pdf', 'authority': 'shop'}


The markdown loader kept the **sections**, so a later answer can cite "returns-policy · 1. The
return window". The PDF only knows **pages**: its headings came back as ordinary lines of text.

PDFs lose more than headings. Here is the shipping policy's rates table, first as written, then as
the PDF hands it back:

In [40]:
ship_md = (POLICIES / "shipping-policy.md").read_text()
print(ship_md[ship_md.index("| Option"):ship_md.index("2.1")])

ship_pdf = ingest.load_pdf(POLICIES / "shipping-policy.pdf")[0].page_content
print(ship_pdf[ship_pdf.index("Option"):ship_pdf.index("2.1")])

| Option | Delivery time (business days) | Price | Free when |
|---|---|---|---|
| Standard | 3–5 | $4.99 | order subtotal is **$75 or more** |
| Express | 1–2 | $12.99 | never |


Option Delivery time (business days) Price Free when
Standard 3–5 $4.99 order subtotal is $75 or more
Express 1–2 $12.99 never



The table's columns are gone. "Standard 3–5 $4.99 order subtotal is $75 or more" is still readable
to you, but nothing in it says which number is the price and which is the days. **A retriever cannot
recover what the loader lost.**

Real-world PDFs are messier still. Part 700 of the US Code of Federal Regulations, one of our two
regulator documents, is printed in two columns and hyphenated across lines:

In [41]:
from pypdf import PdfReader

raw = PdfReader(POLICIES / "cfr-16-part-700.pdf").pages[0].extract_text()
print(raw[:230])
print("\n--- after ingest.clean_pdf_text ---\n")
print(ingest.clean_pdf_text(raw)[:200])

672 
SUBCHAPTER G—RULES, REGULATIONS, STATEMENTS AND  
INTERPRETATIONS UNDER THE MAGNUSON-MOSS WAR -
RANTY ACT  
PART 700—INTERPRETATIONS OF  
MAGNUSON-MOSS WARRANTY  
ACT 
Sec. 
700.1 Products covered.  
700.2 Date of manufacture

--- after ingest.clean_pdf_text ---

672 
SUBCHAPTER G—RULES, REGULATIONS, STATEMENTS AND  
INTERPRETATIONS UNDER THE MAGNUSON-MOSS WARRANTY ACT  
PART 700—INTERPRETATIONS OF  
MAGNUSON-MOSS WARRANTY  
ACT 
Sec. 
700.1 Products covered. 


"WAR - RANTY" is the layout's text, not the author's. Cleaning is part of loading:
`clean_pdf_text` re-joins words split across a line, and does nothing else.

**Choosing a loader:**

| Source | Use | Keeps | Loses |
|---|---|---|---|
| Markdown | split on headings (`MarkdownHeaderTextSplitter`) | sections, tables as text | nothing that matters |
| A clean digital PDF | `pypdf` | text, page numbers | headings, table structure, columns |
| Complex layout, tables, scans | a structure-aware parser (Docling-class, appendix B) | tables, layout, OCR text | speed: it downloads models and is slow |
| Many files | `lazy_load()`, one file at a time | memory | nothing |

Two notes from building this. LangChain's `PyPDFLoader` lives in `langchain-community`, which is being
sunset; LangChain's own knowledge-base tutorial now calls `pypdf` directly, and so do we. PyMuPDF is
better at layout, but it is **AGPL**-licensed, a licence question to answer before a quality one.

## 6. Chunking: the one-variable staircase

A search returns **chunks**, not documents. Too big, and a chunk matches every question weakly. Too
small, and a rule gets cut away from its condition. `chunk_size` is a decision, so we run it as an
experiment: the same returns policy, three sizes, nothing else changed.

The test is clause 1.2's last sentence: *"Unopened audio products follow the 30-day window in
1.1."* Does it stay in one piece?

In [42]:
whole = [Document(page_content=policy, metadata={"source": "returns-policy"})]
for size in (200, 800, 2000):
    chunks = ingest.chunk(whole, size, size // 5)
    holder = next(c for c in chunks if "Unopened audio" in c.page_content)
    whole_clause = "30-day window" in holder.page_content
    print(f"chunk_size={size:<5} {len(chunks):>2} chunks   exception and its '30-day window' "
          f"in the same chunk: {whole_clause}")
    if not whole_clause:
        print("      the chunk ends: ...", " ".join(holder.page_content.split())[-60:])

chunk_size=200   13 chunks   exception and its '30-day window' in the same chunk: False
      the chunk ends: ... y**, for hygiene reasons. Unopened audio products follow the
chunk_size=800    3 chunks   exception and its '30-day window' in the same chunk: True
chunk_size=2000   1 chunks   exception and its '30-day window' in the same chunk: True


<img src="figures/chunking-staircase.png" width="900"
     alt="The chunking staircase. The same returns policy is split at three chunk sizes. At 200
     characters the sentence about unopened audio products is cut from its 30-day number, so a
     search finds a rule with no number attached. At 800 the clause stays whole. At 2000 the whole
     policy is one blob that matches every question weakly. Overlap is drawn as a shared seam
     between neighbouring chunks.">

At **200**, the chunk ends *"Unopened audio products follow the"*, and the **30-day** it refers to
is in the next chunk. A search can now find the rule without its number. At **800**, the clause is
whole. At **2000**, the whole policy is one chunk, which matches every returns question equally, so
it ranks for none of them in particular.

`chunk_overlap` (a fifth of the size here) repeats the end of each chunk at the start of the next.
It is insurance at the seams, not a fix: at 200 the seam still landed in the wrong place.

One more thing changes the picture. `kb/ingest.py` splits markdown **on its headings first**, so no
chunk ever spans two sections, and a section shorter than `chunk_size` stays one chunk whatever the
size. Section 12 scores all three sizes properly. That is where `settings.CHUNK_SIZE = 800` comes
from.

## 7. Metadata: citation raw material

Session 9's answers must say *where* a rule came from. An answer can't cite a section that ingestion
forgot to record, so every chunk leaves ingestion carrying it:

In [43]:
corpus = ingest.load_corpus()
chunks = ingest.chunk(corpus)

examples = {}
for c in chunks:
    if c.metadata["section"] != "preamble":
        examples.setdefault((c.metadata["format"], c.metadata["authority"]), c)
for c in examples.values():
    print(c.metadata)
print(f"\n{len(corpus)} documents -> {len(chunks)} chunks at chunk_size {settings.CHUNK_SIZE}")

{'section': '1. The return window', 'source': 'returns-policy', 'format': 'md', 'authority': 'shop', 'start_index': 0}
{'source': 'ftc-prompt-delivery', 'section': 'p.1', 'page': 1, 'format': 'pdf', 'authority': 'regulator', 'start_index': 0}

24 documents -> 70 chunks at chunk_size 800


| Field | Why it exists |
|---|---|
| `source`, `section` | the citation: *returns-policy · 1. The return window*, or *cfr-16-part-700 · p.4* |
| `page` | PDFs only: the page a person would turn to |
| `start_index` | where in its section the chunk begins; part of its stable id (section 10) |
| `format` | `md` or `pdf`, so you can tell which loader made it |
| `authority` | `shop` or `regulator`. Federal law is not a ShopWise promise, and an answer must never pass one off as the other |

`load_corpus` indexes each ShopWise policy from its **markdown only**. The `.pdf` twins exist for the
loader lesson above. Indexing both would return every clause twice: the "duplicate hits" symptom that
session 9's debugging table traces back to ingestion.

---
# Part B: Embeddings and search, measured

## 8. Embeddings: meaning as coordinates

An embedding model turns text into a list of numbers: coordinates in a space where **similar
meanings land close together**. "Close" is measured by **cosine similarity**, the angle between two
vectors: 1.0 for the same direction, lower as they diverge. It is four lines of numpy:

In [45]:
emb = store.embeddings()


def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


PHRASES = ["refund my order", "give me my money back", "আমার টাকা ফেরত চাই", "when will it ship"]
vectors = {p: emb.embed_query(p) for p in PHRASES}
print("dimensions:", len(vectors[PHRASES[0]]))
for p in PHRASES[1:]:
    print(f"cosine('refund my order', {p!r:24}) = {cosine(vectors[PHRASES[0]], vectors[p]):.3f}")

dimensions: 768
cosine('refund my order', 'give me my money back' ) = 0.799
cosine('refund my order', 'আমার টাকা ফেরত চাই'    ) = 0.693
cosine('refund my order', 'when will it ship'     ) = 0.662


"Refund" and "money back" share no words, yet the paraphrase is clearly nearest: the model matched
**meaning**, not spelling. The Bangla sentence ("I want my money back") lands closer than the
unrelated shipping question, but not by as much. Meaning does cross languages in one space, more
weakly than it travels within one. Read the gaps, not the absolute numbers.

<img src="figures/meaning-map.png" width="900"
     alt="Embeddings as coordinates. On a map of meaning, &quot;refund my order&quot; and &quot;give
     me my money back&quot; sit close together with a Bangla sentence of the same meaning beside
     them, and &quot;when will it ship&quot; sits far away. A second panel shows the same-space
     rule: a query embedded by a different model lands in a different space and returns confident
     noise. The takeaway: index and query with the same model, at the same size, forever, or
     re-index.">

**768 or 3072?** `gemini-embedding-2` returns 3072 numbers by default and can be cut to 768 with
almost no loss: it is trained so that the first numbers carry the most meaning (*Matryoshka*
embeddings). The same comparisons at full size:

In [46]:
from langchain.embeddings import init_embeddings

full = init_embeddings(f"google_genai:{settings.embed_name}")      # the default size
full_vectors = {p: full.embed_query(p) for p in PHRASES}
print("dimensions:", len(full_vectors[PHRASES[0]]))
for p in PHRASES[1:]:
    print(f"cosine('refund my order', {p!r:24}) = {cosine(full_vectors[PHRASES[0]], full_vectors[p]):.3f}")

dimensions: 3072
cosine('refund my order', 'give me my money back' ) = 0.792
cosine('refund my order', 'আমার টাকা ফেরত চাই'    ) = 0.686
cosine('refund my order', 'when will it ship'     ) = 0.636


The order is unchanged and the storage is a quarter. That is why `settings.EMBED_DIM` is 768.

## 9. The same-space rule

**A query is only comparable with vectors from the same model at the same size.** Index with one and
query with another, and the coordinates mean different things. There are two ways to break it:

| You change | What happens |
|---|---|
| the dimension | Qdrant refuses: the sizes don't match. The lucky case |
| the model, same dimension | nothing errors. Results come back with confident scores, and they are noise (exercise 4 makes this happen) |

The loud case first: index at 768, then query at 1536. Same model, different size.

In [47]:
lab = store.open_client(":memory:")                                 # a throwaway experiment
store.ensure_collection(lab, emb, "same_space_demo")
store.vector_store(lab, emb, name="same_space_demo").add_texts(["Gift cards cannot be returned."])

wrong_size = init_embeddings(f"google_genai:{settings.embed_name}", output_dimensionality=1536)
try:
    lab.query_points("same_space_demo", query=wrong_size.embed_query("gift card refund"), using=store.DENSE)
except Exception as exc:
    print(type(exc).__name__, "->", str(exc)[:110])

ValueError -> shapes (1,768) and (1536,) not aligned: 768 (dim 1) != 1536 (dim 0)


The defence is a **name**. The collection is named after the space that filled it, so changing the
model changes the collection, and a query can never meet vectors from another space:

In [48]:
os.environ["SHOPWISE_EMBED_MODEL"] = "fastembed:BAAI/bge-small-en-v1.5"
print("with the fallback:", Settings().collection)
del os.environ["SHOPWISE_EMBED_MODEL"]
print("default:          ", Settings().collection)

with the fallback: policies__bge-small-en-v1.5
default:           policies__gemini-embedding-2-768


**No Gemini key?** Put `SHOPWISE_EMBED_MODEL=fastembed:BAAI/bge-small-en-v1.5` in `.env` and
run this notebook from the top. A small model runs on your CPU (about 70 MB, downloaded once), in its
own collection. Two cells will fail, both on purpose: the 3072-dimension comparison in section 8 and
the 1536 one above, which ask Gemini for other sizes. Everything else works. The switch is
deliberate, never automatic: a quiet fallback would let two teammates query two different spaces
through the same code, and the only symptom would be bad answers.

## 10. Qdrant, on disk

A **collection** is a vector size, a distance metric and a set of points. Ours lives in
`.state/qdrant/`: on disk, so session 9 opens the same one, and gitignored, because it is
rebuilt from the corpus in seconds. It runs **embedded**, inside this Python process: no server, no
signup, no account.

Two details carry the weight. The vector size is **probed** from a real embedding, never typed. And
every chunk's id is derived from where it came from:

In [49]:
show("chunk_id")

def chunk_id(doc: Document) -> str:
    m = doc.metadata
    return str(uuid.uuid5(NAMESPACE, f"{m['source']}#{m['section']}#{m['start_index']}"))


`uuid5` is a hash: the same text always gives the same id. So the id of "returns-policy, section 1,
character 0" is the same today, tomorrow and on your teammate's laptop. Write the corpus twice:

In [50]:
client = store.open_client()                      # .state/qdrant
size = store.ensure_collection(client, emb)
writer = store.vector_store(client, emb, mode="hybrid")
ids = [ingest.chunk_id(c) for c in chunks]

print(f"{len(chunks)} chunks, vector size {size}")
for attempt in (1, 2):
    writer.add_documents(chunks, ids=ids)
    print(f"write {attempt}: {client.count(settings.collection).count} points stored")

70 chunks, vector size 768
write 1: 70 points stored
write 2: 70 points stored


Written twice, stored once. The same chunk gets the same id, so the second write **overwrites** each
point with itself. Re-running a cell, or the whole script, never duplicates the knowledge base.
(Change `CHUNK_SIZE`, though, and the chunks themselves change, so their ids change too. That is what
`ingest.py --reset` is for.)

`mode="hybrid"` means each chunk is written with **two** vectors: the embedding, and a keyword vector
computed on your CPU. Tonight searches only the first. Session 9 uses both, and a collection's vector
layout is fixed when it is created, so it is declared now.

**The lock.** Embedded Qdrant allows one client per folder. This notebook now holds it, so a second
client, or `ingest.py` in a terminal, is refused:

In [28]:
try:
    store.open_client()
except RuntimeError as exc:
    print("RuntimeError:", str(exc).replace(str(settings.QDRANT_PATH), ".state/qdrant").split(" If")[0])

RuntimeError: Storage folder .state/qdrant is already accessed by another instance of Qdrant client.


The fix is `client.close()` (or restarting the kernel) before another program opens the folder.
Section 13 does exactly that.

## 11. Search, and inspect the evidence

`hits` prints each result as score · source · section · the start of the text. Read the chunks, not
just the scores. First, the gotcha session 9 has to get right: opened headphones.

In [53]:
dense = store.vector_store(client, emb)


def hits(query, k=4, filter=None):
    """Top-k results, one line each."""
    print(query)
    for doc, score in dense.similarity_search_with_score(query, k=k, filter=filter):
        m = doc.metadata
        print(f"  {score:.3f}  {m['source']} · {m['section']:<32} {' '.join(doc.page_content.split())}")


hits("Can I return opened headphones?", k=2)

Can I return opened headphones?
  0.813  returns-policy · 1. The return window             ## 1. The return window 1.1 You may return most items within **30 days of the delivery date** shown in your order history. The window is counted from delivery, not from the order date. 1.2 **Opened audio products** — headphones, earbuds and speakers whose seal has been broken — may be returned within **15 days of delivery**, for hygiene reasons. Unopened audio products follow the 30-day window in 1.1. 1.3 Items must be unused, in their original packaging, with all accessories included.
  0.707  returns-policy · 4. Starting a return             ## 4. Starting a return 4.1 Reply to your order confirmation email, or write to support, with your order number and the reason for the return. The returns team replies within one working day with a return label and instructions. 4.2 Pack the item securely. Items damaged in return transit because of poor packaging may be refused.


The chunk that comes first must be "1. The return window", because that is where clause 1.2 lives:
**opened audio, 15 days**. The generic 30-day rule is in the same section, so if a later answer says
"30 days" for opened headphones, the fault is not retrieval's. Retrieval handed over the exception;
generation would have to drop it. That separation is the point of tonight.

Three more, including one only a regulator answers:

In [54]:
hits("do you deliver to Malaysia?", k=2)
hits("I dropped my kettle and it cracked", k=2)
hits("How long must a seller ship if no time is stated?", k=2)

do you deliver to Malaysia?
  0.714  shipping-policy · 1. Where we ship                 ## 1. Where we ship 1.1 ShopWise ships to addresses within the **United States only**, including PO boxes. 1.2 We do **not** ship internationally, and we cannot forward orders to freight-forwarding services.
  0.662  shipping-policy · 2. Delivery options and rates    ## 2. Delivery options and rates | Option | Delivery time (business days) | Price | Free when | |---|---|---|---| | Standard | 3–5 | $4.99 | order subtotal is **$75 or more** | | Express | 1–2 | $12.99 | never | 2.1 Delivery times are counted from dispatch, not from the order date. Orders placed before 2 pm Eastern on a business day are dispatched the same day; later orders are dispatched the next business day. 2.2 Express is not available to PO boxes.
I dropped my kettle and it cracked
  0.650  warranty-policy · 3. Making a claim                ## 3. Making a claim 3.1 Write to support with your **order number** and a **photo or short 

Read the kettle result closely. The answer (accidental damage is **not covered**) is in the warranty
policy's "what the warranty covers" section. Check whether it came first or second. Found but not
first is a real difference, and section 12's second number, MRR, measures exactly that.

**Scores are relative.** 0.8 means nothing on its own: it depends on the model, the dimension and the
corpus. Compare scores within one search; never copy a threshold from another system.

### 11.1 Filters are rules, not rankings

A filter keeps a source out of the search entirely; it is never ranked in. The same question, asked of
the shop and of the regulator:

In [55]:
for who in ("shop", "regulator"):
    doc = dense.similarity_search("how long until my order ships?", k=1,
                                  filter=store.where(authority=who))[0]
    print(f"{who:9} -> {doc.metadata['source']} · {doc.metadata['section']}: "
          f"{' '.join(doc.page_content.split())[:100]}")

shop      -> shipping-policy · 2. Delivery options and rates: ## 2. Delivery options and rates | Option | Delivery time (business days) | Price | Free when | |---
regulator -> ftc-prompt-delivery · p.1: a reasonable basis for believing that you can ship within 30 days. If you can’t ship within the prom


The shop's answer is ShopWise's promise. The regulator's is federal law about sellers in general. A
support answer must say which one it is quoting, and session 9's citations carry `authority` for
exactly that reason.

## 12. Measure retrieval, before any answer exists

Reading hits aloud doesn't scale past four questions, and it doesn't survive a change.
`data/kb_questions.yml` holds 19 gold questions: 15 answerable, each naming the source that answers
it and **evidence** phrases that must all appear in one chunk, and 4 the corpus deliberately can't
answer (session 9's refusals).

| Metric | Question it answers |
|---|---|
| **hit-rate@k** | did a chunk holding the evidence appear in the top k? |
| **MRR** | how high did it appear, on average? 1 for first, ½ for second, 0 for never |

The scorer is ten lines and needs no judge:

In [ ]:
show("is_gold", "kb/retrieval_eval.py")
show("score", "kb/retrieval_eval.py")

def is_gold(q: dict, doc: Document) -> bool:
    """From the expected source, and holding every evidence phrase.

    Whitespace is collapsed first: the markdown wraps lines, so "free return
    shipping" can arrive as "free return\\nshipping"."""
    text = " ".join(doc.page_content.split()).lower()
    return (doc.metadata.get("source") == q["source"]
            and all(e.lower() in text for e in q["evidence"]))
def score(search: Search, k: int = 4) -> dict:
    """Run every answerable gold question through `search(question, k)`."""
    ranks = {q["id"]: rank_of(q, search(q["question"], k)) for q in gold_questions()}
    n = len(ranks)
    return {"k": k,
            "hit_rate": sum(r is not None for r in ranks.values()) / n,
            "mrr": sum(1 / r for r in ranks.values() if r) / n,
            "ranks": ranks}


Now the staircase from section 6, scored. Three chunk sizes, each built in a throwaway in-memory
collection so the real knowledge base is untouched. This cell embeds the corpus three times, about
350 chunks, so it takes a minute.

In [33]:
print(f"{'chunk_size':>10} {'chunks':>6} {'hit@1':>6} {'hit@4':>6} {'MRR':>5}  misses at k=4")
for size in (200, 800, 2000):
    lab = store.open_client(":memory:")
    name = f"staircase_{size}"
    store.ensure_collection(lab, emb, name)
    sized = ingest.chunk(corpus, size, size // 5)
    vs = store.vector_store(lab, emb, name=name)
    vs.add_documents(sized, ids=[ingest.chunk_id(c) for c in sized])
    search = lambda q, k: vs.similarity_search(q, k=k)
    at1, at4 = scorer.score(search, 1), scorer.score(search, 4)
    misses = [q for q, r in at4["ranks"].items() if r is None]
    print(f"{size:>10} {len(sized):>6} {at1['hit_rate']:>6.2f} {at4['hit_rate']:>6.2f}"
          f" {at4['mrr']:>5.2f}  {', '.join(misses)}")
    lab.close()

chunk_size chunks  hit@1  hit@4   MRR  misses at k=4
       200    253   0.80   0.87  0.83  KBQ-05, KBQ-15
       800     70   0.87   1.00  0.92  
      2000     40   0.87   1.00  0.92  


How to read the table, and what to say only if the table says it:

- **200 is too small** if it misses questions the other two find. Those misses are clauses whose
  condition was cut away, section 6's failure, now counted.
- **800 and 2000 tie or nearly tie.** Headings already cap most chunks at one section, so above about
  800 the size barely matters on this corpus. 800 wins on **cost**: in session 9 every retrieved chunk
  is pasted into a prompt, and an 800-character chunk costs less than half of a 2000-character one.
- **hit@1 vs hit@4** is the gap between "found it first" and "found it somewhere". Session 9's `k` is
  chosen from that gap.

This is session 3's habit at retrieval scale: change one thing, watch the number. And it comes
**first**, because generation cannot fix what retrieval did not find.

## 13. v5: what you built

<img src="figures/v5-architecture.png" width="900"
     alt="The v5 architecture. kb/ingest.py loads three ShopWise policies in markdown and two
     regulator PDFs, splits text on headings and then by size, embeds it with gemini-embedding-2 at
     768 dimensions, and writes it into one Qdrant collection on disk, named after its embedding
     space, with stable ids so a second run changes nothing. A test query, embedded with the same
     model, returns scored hits carrying source, section, page and authority, and is scored with
     hit-rate and MRR. The v4 agent is shown beside it, not yet connected: session 9 wires it.">

| File | Change | Why it is separate |
|---|---|---|
| `kb/ingest.py` | **new**: load → chunk → embed → write, stable ids, `--reset` | building the knowledge base is a job you rerun whenever a policy changes |
| `kb/store.py` | **new**: embeddings, client, collection, filters | one door: the notebook, the script and session 9's tool can't disagree about the space |
| `kb/retrieval_eval.py` | **new**: hit-rate@k and MRR against the gold questions | a regression check any later change can rerun |
| `test_kb.py` | **new**: loading, ids, idempotency, filters, the scorer, with a fake embedding | the plumbing tested with no key and no network |
| `config.py` | `EMBED_DIM`, `QDRANT_PATH`, `CHUNK_SIZE`, `CHUNK_OVERLAP`, derived `collection`; 5.0.0 | one `Settings`, as since session 3 |
| `data/` | `policies/`, `kb_questions.yml` | shared fixtures, not snapshot code |

The agent is **unchanged**. v5 is a library the agent doesn't know about yet.

The script, run twice from a terminal, after closing this notebook's client because of the lock:

In [34]:
client.close()                                 # release .state/qdrant for the script

for run in (1, 2):
    out = subprocess.run([sys.executable, "kb/ingest.py"], cwd=HERE / "project",
                         capture_output=True, text=True).stdout
    print(f"--- run {run} ---\n{out.replace(str(settings.QDRANT_PATH), '.state/qdrant')}")

--- run 1 ---
embedding space : gemini-embedding-2 · 768
collection      : policies__gemini-embedding-2-768
folder          : .state/qdrant
chunks          : 70  (chunk_size 800, overlap 160)
points stored   : 70
smoke query     : returns-policy · 1. The return window

--- run 2 ---
embedding space : gemini-embedding-2 · 768
collection      : policies__gemini-embedding-2-768
folder          : .state/qdrant
chunks          : 70  (chunk_size 800, overlap 160)
points stored   : 70
smoke query     : returns-policy · 1. The return window



Same point count both times. In a terminal, from `session-08/project/`:

```bash
uv run python kb/ingest.py              # build or refresh the knowledge base
uv run python kb/retrieval_eval.py      # hit-rate@4 and MRR
uv run python test_kb.py                # the plumbing, no key needed
```

## 14. Where this leaves us

| v4 | v5 |
|---|---|
| policy answers from memory | ShopWise policies and regulator PDFs in a knowledge base on disk |
| — | ingestion you can rerun safely, with the embedding space named in the collection |
| — | every chunk carries source, section, page and authority |
| — | retrieval scored, hit-rate@k and MRR, before any answer exists |

> Ask v4 about Priya's backpack again, and nothing has changed: the library is stocked, and the agent
> has never heard of it. Search results don't answer customers; grounded answers do. Next session: a
> librarian the agent can call, citations, honest refusals, and a scorecard for the answers.

Exercises: `exercises.ipynb`.

### 14.1 Interview questions

1. *Fine-tune or RAG for a support bot whose refund rules change monthly?* RAG. Fine-tuning teaches
   behaviour, not facts, and a monthly retrain is slower and dearer than editing a document.
2. *You switched embedding models and search went to noise, with no error. What happened?* The index
   holds vectors from the old model; the query is in the new model's space. Same size, so nothing
   complained. Recovery is re-embedding the whole corpus, and naming collections after their space
   so it can't happen silently again.
3. *Symptoms of chunks too small, and too large?* Too small: answers with a number and no condition,
   or a rule with no number. Too large: every question retrieves the same few blobs, and prompts get
   long and expensive.
4. *Why score retrieval before generation?* Generation can't use a passage it was never given. If
   the answer is wrong, the first question is whether the right chunk was retrieved at all.

## Appendix: for afterwards

### A. Photos in the same space

`gemini-embedding-2` is **multimodal**: it puts an image into the same space as text. A customer's
photo of broken headphones lands near the catalogue photo of SKU-100 and near the words
"noise-cancelling headphones". LangChain's embeddings interface is text-only, so images go through the
Google SDK (`client.models.embed_content`). One gotcha: several images in one `contents` list come
back as **one averaged vector** unless each sits in its own `types.Content`. And in a mixed search,
text-to-image similarity runs systematically lower than text-to-text, so always filter by kind.

### B. Structure-aware parsing

Docling-class parsers rebuild tables, reading order and headings from complex PDFs and scans, often
with OCR. They are slow (model downloads, CPU-heavy), so cache what they produce and never parse
inside a request.

### C. MMR and score thresholds

`dense.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20})` trades a little
relevance for **diversity**, so four near-identical chunks don't crowd out a different one.
`search_type="similarity_score_threshold"` drops weak hits, but the threshold only means something for
this model, size and corpus (section 11).

### D. The fallback, in full

`fastembed:BAAI/bge-small-en-v1.5` scores close to Gemini on this corpus, but it is English-only: the
Bangla example in section 8 does not survive the switch, which is a lesson in itself. A cheap paid
route is OpenAI's `text-embedding-3-small` if a team already has that key.

### E. Keyword and hybrid search (session 9)

The collection already holds a keyword (BM25) vector for every chunk. `store.vector_store(client,
mode="keyword")` searches by exact words; `mode="hybrid"` fuses both rankings. Session 9 measures
whether that helps on this corpus, where exact codes like `ORD-5004` matter.

### F. Case study: the messy corpus

A client hands you 40 product manuals (clean PDFs), an FAQ web page, and 200 scanned warranty cards.
Write a one-page ingestion plan: the loader, chunking and metadata for each source; what you refuse to
promise on the scans without an OCR budget; and how you will score retrieval before anyone sees an
answer. This is genuinely billable junior work.